# Baseball Win Probability — Experiment Comparison

Runs multiple logistic regression configurations and compares results side-by-side.  
Train: 2023–2024 seasons | Test: 2025 season

To add a new experiment, append a dict to `EXPERIMENTS` in the config cell and re-run.

In [ ]:
from pathlib import Path

# ── Data ────────────────────────────────────────────────────────────────────
DATA_DIR      = Path("data")
TRAIN_PATH    = DATA_DIR / "train_features.parquet"
TEST_PATH     = DATA_DIR / "test_features.parquet"
METADATA_COLS = ["date", "home_team", "away_team", "home_win"]

# ── Experiments ──────────────────────────────────────────────────────────────
# Each dict defines one full pipeline run. Keys:
#   name              : label shown in all plots and tables
#   use_differential  : True = 59 home-minus-away diff features; False = 118 raw features
#   use_scale         : True = StandardScaler before model (and before poly if enabled)
#   poly_degree       : None = no expansion; 2 = degree-2 polynomial features
#   poly_interaction  : True = interaction terms only (no x^2); ignored if poly_degree=None
#   model             : kwargs passed directly to sklearn LogisticRegression
EXPERIMENTS = [
    {
        "name":             "Baseline (diff, L2 C=1)",
        "use_differential": True,
        "use_scale":        False,
        "poly_degree":      None,
        "poly_interaction": True,
        "model": {"penalty": "l2", "C": 1.0, "max_iter": 10000, "random_state": 42},
    },
    {
        "name":             "Poly interactions (diff, L2 C=0.1)",
        "use_differential": True,
        "use_scale":        False,
        "poly_degree":      2,
        "poly_interaction": True,
        "model": {"penalty": "l2", "C": 0.1, "max_iter": 10000, "random_state": 42},
    },
    {
        "name":             "Poly interactions (diff, L2 C=0.01)",
        "use_differential": True,
        "use_scale":        False,
        "poly_degree":      2,
        "poly_interaction": True,
        "model": {"penalty": "l2", "C": 0.01, "max_iter": 10000, "random_state": 42},
    },
    {
        "name":             "L1 sparse (diff, no poly)",
        "use_differential": True,
        "use_scale":        False,
        "poly_degree":      None,
        "poly_interaction": True,
        "model": {"penalty": "l1", "C": 1.0, "solver": "liblinear", "max_iter": 10000, "random_state": 42},
    },
    {
        "name":             "Raw features (no diff, L2 C=1)",
        "use_differential": False,
        "use_scale":        False,
        "poly_degree":      None,
        "poly_interaction": True,
        "model": {"penalty": "l2", "C": 1.0, "max_iter": 10000, "random_state": 42},
    },
]

# ── Drill-down ───────────────────────────────────────────────────────────────
# Name of experiment to show detailed feature importance + calibration plots for
INSPECT_EXPERIMENT = "Poly interactions (diff, L2 C=0.1)"

# ── Evaluation ───────────────────────────────────────────────────────────────
CALIBRATION_BINS = 10
TOP_N_FEATURES   = 20

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    brier_score_loss,
)
from sklearn.calibration import calibration_curve

sns.set_theme(style="whitegrid")

## 2. Load Data

In [ ]:
train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)

print(f"Train: {train_df.shape}  ({train_df['date'].dt.year.min()}-{train_df['date'].dt.year.max()})")
print(f"Test:  {test_df.shape}   ({test_df['date'].dt.year.min()}-{test_df['date'].dt.year.max()})")
print(f"\nTarget balance (train): {train_df['home_win'].mean():.3f} home win rate")
print(f"Target balance (test):  {test_df['home_win'].mean():.3f} home win rate")

feat_cols = [c for c in train_df.columns if c not in METADATA_COLS]
y_train   = train_df["home_win"].values
y_test    = test_df["home_win"].values

## 3. Pipeline Helpers

In [ ]:
BATTING_STATS            = ["BA", "OBP", "SLG", "OPS", "K%", "BB%"]
PITCHING_STATS           = ["ERA", "WHIP", "SO9", "SO/W", "IP"]
PITCHING_LOWER_IS_BETTER = {"ERA", "WHIP"}


def make_differential_features(df):
    """Return a DataFrame of home-minus-away differential features.

    Batting:  diff_bat{N}_{stat} = home_bat{N}_{stat} - away_bat{N}_{stat}
    Pitching: diff_sp_{stat}     = home_sp_{stat} - away_sp_{stat}   (SO9, SO/W, IP)
              diff_sp_{stat}     = away_sp_{stat} - home_sp_{stat}   (ERA, WHIP)
    All columns oriented so positive = home team advantage.
    """
    diffs = {}
    for n in range(1, 10):
        for stat in BATTING_STATS:
            diffs[f"diff_bat{n}_{stat}"] = df[f"home_bat{n}_{stat}"] - df[f"away_bat{n}_{stat}"]
    for stat in PITCHING_STATS:
        if stat in PITCHING_LOWER_IS_BETTER:
            diffs[f"diff_sp_{stat}"] = df[f"away_sp_{stat}"] - df[f"home_sp_{stat}"]
        else:
            diffs[f"diff_sp_{stat}"] = df[f"home_sp_{stat}"] - df[f"away_sp_{stat}"]
    return pd.DataFrame(diffs, index=df.index)


def run_experiment(cfg, train_df, test_df, feat_cols, y_train, y_test):
    """Run one full pipeline (features → preprocessing → train → evaluate).

    Returns a dict with metrics, predictions, model, and feature names
    needed for downstream plots.
    """
    # --- Feature engineering ---
    if cfg["use_differential"]:
        X_tr = make_differential_features(train_df)
        X_te = make_differential_features(test_df)
    else:
        X_tr = train_df[feat_cols].copy()
        X_te = test_df[feat_cols].copy()

    assert X_tr.isna().sum().sum() == 0
    assert X_te.isna().sum().sum() == 0

    base_feature_names = list(X_tr.columns)

    # --- Preprocessing ---
    if cfg["use_scale"]:
        scaler = StandardScaler()
        X_tr_arr = scaler.fit_transform(X_tr)
        X_te_arr = scaler.transform(X_te)
    else:
        X_tr_arr = X_tr.values
        X_te_arr = X_te.values

    feature_names = base_feature_names

    if cfg["poly_degree"] is not None:
        poly = PolynomialFeatures(
            degree=cfg["poly_degree"],
            interaction_only=cfg["poly_interaction"],
            include_bias=False,
        )
        X_tr_arr = poly.fit_transform(X_tr_arr)
        X_te_arr = poly.transform(X_te_arr)
        feature_names = poly.get_feature_names_out(base_feature_names).tolist()

    # --- Train ---
    model = LogisticRegression(**cfg["model"])
    model.fit(X_tr_arr, y_train)

    if model.n_iter_[0] == cfg["model"].get("max_iter", 100):
        print(f"  WARNING [{cfg['name']}]: max_iter reached ({model.n_iter_[0]})")

    # --- Evaluate ---
    y_pred_tr = model.predict(X_tr_arr)
    y_pred_te = model.predict(X_te_arr)
    y_prob_tr = model.predict_proba(X_tr_arr)[:, 1]
    y_prob_te = model.predict_proba(X_te_arr)[:, 1]

    return {
        "name":          cfg["name"],
        "model":         model,
        "feature_names": feature_names,
        "y_test":        y_test,
        "y_pred_test":   y_pred_te,
        "y_prob_train":  y_prob_tr,
        "y_prob_test":   y_prob_te,
        "train_acc":     accuracy_score(y_train, y_pred_tr),
        "test_acc":      accuracy_score(y_test,  y_pred_te),
        "train_auc":     roc_auc_score(y_train,  y_prob_tr),
        "test_auc":      roc_auc_score(y_test,   y_prob_te),
        "train_brier":   brier_score_loss(y_train, y_prob_tr),
        "test_brier":    brier_score_loss(y_test,  y_prob_te),
        "n_iter":        model.n_iter_[0],
        "n_features":    X_tr_arr.shape[1],
    }

## 4. Run All Experiments

In [ ]:
results = []
for cfg in EXPERIMENTS:
    print(f"Running: {cfg['name']} ...", end=" ")
    r = run_experiment(cfg, train_df, test_df, feat_cols, y_train, y_test)
    results.append(r)
    print(f"done  (test acc={r['test_acc']:.4f}, auc={r['test_auc']:.4f}, {r['n_features']} features, {r['n_iter']} iters)")

print(f"\nFinished {len(results)} experiments.")

## 5. Comparison Table

In [ ]:
summary = pd.DataFrame([{
    "Experiment":   r["name"],
    "Features":     r["n_features"],
    "Iters":        r["n_iter"],
    "Train Acc":    r["train_acc"],
    "Test Acc":     r["test_acc"],
    "Train AUC":    r["train_auc"],
    "Test AUC":     r["test_auc"],
    "Train Brier":  r["train_brier"],
    "Test Brier":   r["test_brier"],
} for r in results]).set_index("Experiment")

float_cols = [c for c in summary.columns if summary[c].dtype == float]
(
    summary.style
    .highlight_max(subset=["Test Acc", "Test AUC"], color="#c6efce")
    .highlight_min(subset=["Test Brier"],            color="#c6efce")
    .highlight_max(subset=["Train Acc", "Train AUC"], color="#ffeb9c")
    .format("{:.4f}", subset=float_cols)
    .format("{:,}",   subset=["Features", "Iters"])
)

## 6. Overlaid ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for r in results:
    RocCurveDisplay.from_predictions(
        r["y_test"], r["y_prob_test"],
        name=f"{r['name']} (AUC={r['test_auc']:.3f})",
        ax=ax,
    )
ax.plot([0, 1], [0, 1], "k--", label="Random")
ax.set_title("ROC Curves -- All Experiments (Test Set)")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

## 7. Metric Bar Charts

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
names = [r["name"] for r in results]
short_names = [n.split("(")[0].strip() for n in names]  # trim long labels

for ax, (train_key, test_key, title, lower_better) in zip(axes, [
    ("train_acc",   "test_acc",   "Accuracy",    False),
    ("train_auc",   "test_auc",   "ROC-AUC",     False),
    ("train_brier", "test_brier", "Brier Score", True),
]):
    x   = range(len(results))
    bar_w = 0.35
    ax.bar([i - bar_w/2 for i in x], [r[train_key] for r in results], bar_w, label="Train", alpha=0.7)
    ax.bar([i + bar_w/2 for i in x], [r[test_key]  for r in results], bar_w, label="Test",  alpha=0.7)
    ax.set_xticks(list(x))
    ax.set_xticklabels(short_names, rotation=25, ha="right", fontsize=8)
    ax.set_title(title)
    note = "(lower=better)" if lower_better else "(higher=better)"
    ax.set_ylabel(f"{title} {note}")
    ax.legend()

plt.tight_layout()
plt.show()

## 8. Drill-down: Feature Importance

Showing detailed plots for the experiment named in `INSPECT_EXPERIMENT`.

In [ ]:
r = next((x for x in results if x["name"] == INSPECT_EXPERIMENT), None)
if r is None:
    raise ValueError(f"No experiment named {INSPECT_EXPERIMENT!r}. Check INSPECT_EXPERIMENT in the config cell.")

print(f"Inspecting: {r['name']}")
print(f"  {r['n_features']} features | {r['n_iter']} iterations")
print(f"  Test Acc={r['test_acc']:.4f}  AUC={r['test_auc']:.4f}  Brier={r['test_brier']:.4f}")

coefs = r["model"].coef_[0]
coef_df = (
    pd.DataFrame({"feature": r["feature_names"], "coefficient": coefs})
    .assign(abs_coef=lambda d: d["coefficient"].abs())
    .sort_values("abs_coef", ascending=False)
    .head(TOP_N_FEATURES)
    .sort_values("coefficient")
)

fig, ax = plt.subplots(figsize=(9, 6))
colors = ["#d73027" if c < 0 else "#1a9850" for c in coef_df["coefficient"]]
ax.barh(coef_df["feature"], coef_df["coefficient"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Coefficient (log-odds)")
ax.set_title(f"Top {TOP_N_FEATURES} Coefficients — {r['name']}")
plt.tight_layout()
plt.show()

print(f"\nTop 10 features by absolute coefficient:")
print(coef_df.tail(10)[["feature", "coefficient"]].to_string(index=False))

## 9. Drill-down: Calibration

In [ ]:
# Calibration for INSPECT_EXPERIMENT
prob_true, prob_pred = calibration_curve(
    r["y_test"], r["y_prob_test"], n_bins=CALIBRATION_BINS, strategy="uniform"
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.plot(prob_pred, prob_true, "o-", label=r["name"])
ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Fraction of positives (actual home win rate)")
ax.set_title(f"Calibration Plot -- {r['name']}")
ax.legend()

ax2 = axes[1]
ax2.hist(r["y_prob_test"][r["y_test"] == 0], bins=30, alpha=0.6, label="Away Win", density=True)
ax2.hist(r["y_prob_test"][r["y_test"] == 1], bins=30, alpha=0.6, label="Home Win", density=True)
ax2.set_xlabel("Predicted probability of home win")
ax2.set_ylabel("Density")
ax2.set_title("Predicted Probability Distribution by Outcome")
ax2.legend()

plt.tight_layout()
plt.show()

print(f"Brier Score: {r['test_brier']:.4f}  (lower is better; ~0.25 = random)")

## 10. Drill-down: Confusion Matrix & Classification Report

In [ ]:
print(f"=== Classification Report — {r['name']} (Test) ===")
print(classification_report(r["y_test"], r["y_pred_test"], target_names=["Away Win", "Home Win"]))

fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(r["y_test"], r["y_pred_test"])
ConfusionMatrixDisplay(cm, display_labels=["Away Win", "Home Win"]).plot(ax=ax, colorbar=False)
ax.set_title(f"Confusion Matrix -- {r['name']}")
plt.tight_layout()
plt.show()